# Block editor — draw and edit vineyard block polygons

Draw new blocks or adjust existing ones against fresh imagery, then write the
result back to `data/blocks.geojson`.

**Workflow**
1. Run the cells down to the map.
2. Toggle layers with the layer control: Esri World Imagery (base), the most
   recent clear Sentinel-2 true-color scene, and its NDVI at 50% opacity.
3. The saved blocks are loaded into the draw layer — use the **edit** tool to
   reshape them, the **polygon** tool to add new ones, the **trash** tool to
   remove. The white outlines underneath always show the last *saved* state.
4. Run the save cell and answer the property prompts. It backs up the old file
   to `blocks.geojson.bak` before writing.
5. Re-run `02_timeseries.ipynb` afterwards so the parquet matches the new geometry.

In [14]:
import json
import shutil
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import geemap

from vigor import extract

BLOCKS_PATH = PROJECT_ROOT / "data" / "blocks.geojson"

extract.init_ee()

In [15]:
blocks = extract.load_blocks(BLOCKS_PATH)

penticton = blocks[blocks["site"].astype(str).str.lower() == "oliver"]
if penticton.empty:
    print("No penticton blocks found - using all blocks for the AOI.")
    penticton = blocks
aoi = extract.to_ee_geometry(penticton.geometry.union_all().buffer(1500))

blocks[["block_id", "site", "variety", "planting_year", "area_ha", "n_pixels"]]

,block_id,site,variety,planting_year,area_ha,n_pixels
0,penticton-ub-b1,penticton,chardonnay,2024,3.306338,322
1,oliver-ub-b1,oliver,Merlot,2023,0.882325,83
2,1,penticton,chard,2024,3.171610,308


In [16]:
scene = extract.latest_clear_scene(aoi)
info = extract.scene_summary(scene, aoi)
print(
    f"Scene {info['scene_id']}  date {info['date']}  "
    f"clear fraction over AOI {info['valid_frac']:.3f}"
)

Scene 20260715T185921_20260715T190728_T10UGV  date 2026-07-15  clear fraction over AOI 1.000


In [17]:
TRUE_COLOR_VIS = {"bands": ["B4", "B3", "B2"], "min": 0.0, "max": 0.3}
NDVI_VIS = {
    "min": 0.0,
    "max": 0.9,
    "palette": [
        "#a50026", "#d73027", "#f46d43", "#fdae61", "#fee08b",
        "#d9ef8b", "#a6d96a", "#66bd63", "#1a9850", "#006837",
    ],
}

m = geemap.Map()
m.add_basemap("Esri.WorldImagery")
m.addLayer(scene, TRUE_COLOR_VIS, f"S2 true color {info['date']}")
m.addLayer(extract.ndvi_layer(scene), NDVI_VIS, f"S2 NDVI {info['date']}", True, 0.5)
m.addLayer(extract.outlines_layer(blocks), {}, "saved blocks (white)")
m.centerObject(aoi, 15)

if m.draw_control is None:
    m.add_draw_control()

# Seed the saved features into the draw control so they can be edited in place.
m.draw_control.data = json.loads(BLOCKS_PATH.read_text(encoding="utf-8"))["features"]
m

Map(center=[49.20484153583139, -119.53816035452421], controls=(WidgetControl(options=['position', 'transparent…

## Save

Writes **exactly what the draw layer holds** back to `data/blocks.geojson`
(edits, additions, and deletions included). Every feature must carry the
required properties — `block_id, site, variety, planting_year, baseline_start`
— and you will be prompted for any that are missing. Press Enter to accept the
`[default]` shown. Properties of untouched blocks are preserved; if the draw
tool drops them during an edit, they are recovered by matching the unchanged
geometry against the saved file.

In [18]:
REQUIRED_PROPS = ["block_id", "site", "variety", "planting_year", "baseline_start"]
INT_PROPS = {"planting_year", "baseline_start"}


def _clean_props(props):
    """Strip stray whitespace from keys and drop draw-tool style junk."""
    return {
        k.strip(): v
        for k, v in (props or {}).items()
        if k.strip() and k.strip() != "style" and v not in (None, "")
    }


def _default_for(key, props):
    if key == "site":
        return "penticton"
    if key == "baseline_start":
        try:
            return max(2019, int(props["planting_year"]) + 1)
        except (KeyError, TypeError, ValueError):
            return 2019
    return None


def _geom_key(geometry):
    return json.dumps(geometry, sort_keys=True)


drawn = [
    f for f in m.draw_control.data
    if f.get("geometry", {}).get("type") in ("Polygon", "MultiPolygon")
]
if not drawn:
    raise RuntimeError("Draw layer is empty - nothing to save.")

# Recover properties for features the draw tool round-tripped without them.
saved = json.loads(BLOCKS_PATH.read_text(encoding="utf-8"))["features"]
saved_props = {_geom_key(f["geometry"]): _clean_props(f.get("properties")) for f in saved}

features = []
for i, feat in enumerate(drawn):
    props = _clean_props(feat.get("properties"))
    if "block_id" not in props:
        props = {**saved_props.get(_geom_key(feat["geometry"]), {}), **props}
    label = props.get("block_id", f"feature {i + 1}/{len(drawn)} (new)")
    for key in REQUIRED_PROPS:
        if props.get(key) in (None, ""):
            default = _default_for(key, props)
            hint = f" [{default}]" if default is not None else ""
            answer = input(f"{label} - {key}{hint}: ").strip()
            value = answer if answer else default
            if value in (None, ""):
                raise ValueError(f"{key} is required for {label}")
            props[key] = value
            if key == "block_id":
                label = props["block_id"]
    for key in INT_PROPS:
        props[key] = int(props[key])
    features.append(
        {"type": "Feature", "properties": props, "geometry": feat["geometry"]}
    )

ids = [f["properties"]["block_id"] for f in features]
duplicates = sorted({b for b in ids if ids.count(b) > 1})
if duplicates:
    raise ValueError(f"Duplicate block_id values: {duplicates}")

backup = BLOCKS_PATH.with_suffix(".geojson.bak")
shutil.copy2(BLOCKS_PATH, backup)
BLOCKS_PATH.write_text(
    json.dumps({"type": "FeatureCollection", "features": features}, indent=2),
    encoding="utf-8",
)
print(f"Wrote {len(features)} blocks -> {BLOCKS_PATH}")
print(f"Backup of previous file -> {backup}")
print("Re-run notebooks/02_timeseries.ipynb to refresh the parquet.")

feature 4/4 (new) - block_id:  oliver-ub-b1
oliver-ub-b1 - site [penticton]:  oliver
oliver-ub-b1 - variety:  Merlot
oliver-ub-b1 - planting_year:  2023
oliver-ub-b1 - baseline_start [2024]:  2024


ValueError: Duplicate block_id values: ['oliver-ub-b1']